In [ ]:
!pip install torch-geometric torch-scatter torch-sparse
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 20.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.9 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/usr/l

ModuleNotFoundError: No module named 'torch_geometric'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.5.0+cu121.html

  Using cached torch_geometric-2.8.0-py3-none-any.whl.metadata (64 kB)
Using cached torch_geometric-2.8.0-py3-none-any.whl (1.3 MB)
Looking in links: https://data.pyg.org/whl/torch-2.5.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 49.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 135.6 MB/s eta 0:00:00


In [ ]:
# Load verified features
node_features = pd.read_csv("/content/drive/MyDrive/AI_Crime_Prediction/datasets/GCN_Node_Features.csv")

# Create lookup
grid_to_idx = {grid_id: i for i, grid_id in enumerate(node_features["Grid_ID"])}

# Build edges (8-neighbourhood)
edges = []
for i in range(len(node_features)):
    for j in range(i + 1, len(node_features)):
        if abs(node_features.iloc[i]["Grid_X"] - node_features.iloc[j]["Grid_X"]) <= 1 and \
           abs(node_features.iloc[i]["Grid_Y"] - node_features.iloc[j]["Grid_Y"]) <= 1:
            u, v = grid_to_idx[node_features.iloc[i]["Grid_ID"]], grid_to_idx[node_features.iloc[j]["Grid_ID"]]
            edges.extend([[u, v], [v, u]])

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [ ]:
from torch_geometric.data import Data # Ensure this is imported here

# Features: exclude ID, Coordinates, and Target
feature_cols = ['Hawkes_Intensity', 'VICTIM COUNT', 'Accused Count',
                'Unique_Crime_Types', 'Crime_Count']

X = torch.tensor(node_features[feature_cols].values, dtype=torch.float)
y = torch.tensor(node_features['Risk_Label'].values, dtype=torch.long)

# Create the graph_data object
graph_data = Data(x=X, edge_index=edge_index, y=y)

# Masking for training/testing
num_nodes = graph_data.num_nodes
train_size = int(0.8 * num_nodes)
indices = torch.randperm(num_nodes)
graph_data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
graph_data.train_mask[indices[:train_size]] = True
graph_data.test_mask = ~graph_data.train_mask

print("Graph data created successfully!")
print(graph_data)

Graph data created successfully!
Data(x=[2825, 5], edge_index=[2, 10364], y=[2825], train_mask=[2825], test_mask=[2825])


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv  # <--- THIS IS THE MISSING LINK

class CrimeGCN(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 32)
        self.conv2 = GCNConv(32, 16)
        self.fc = torch.nn.Linear(16, 3)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.fc(x)

# Initialize model and optimizer
model = CrimeGCN(in_channels=len(feature_cols))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

# Training loop
for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    out = model(graph_data)
    # Ensure we use train_mask
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Save the model
torch.save(model.state_dict(), "/content/drive/MyDrive/AI_Crime_Prediction/models/gcn_model.pth")
print("GCN Model Saved Successfully!")

Epoch 10, Loss: 0.9015
Epoch 20, Loss: 0.7295
Epoch 30, Loss: 0.4888
Epoch 40, Loss: 0.4091
Epoch 50, Loss: 0.3838
GCN Model Saved Successfully!


In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(graph_data)
    predictions = outputs.argmax(dim=1)

from sklearn.metrics import classification_report
print(classification_report(graph_data.y[graph_data.test_mask], predictions[graph_data.test_mask]))

              precision    recall  f1-score   support

           0       0.90      0.97      0.93       404
           1       0.65      0.37      0.47        86
           2       0.67      0.72      0.69        75

    accuracy                           0.84       565
   macro avg       0.74      0.69      0.70       565
weighted avg       0.83      0.84      0.83       565

